# **PROCES OF MERGERING IN SQL**

Articles looked at to see what vitals/labs essential to mortality and readmission:
- https://pmc.ncbi.nlm.nih.gov/articles/PMC9222812/ 

- https://www.nature.com/articles/s41598-020-78184-7

- https://pmc.ncbi.nlm.nih.gov/articles/PMC10796141/ 

- https://www.frontiersin.org/journals/cardiovascular-medicine/articles/10.3389/fcvm.2025.1590367/full


## EVERYTHING DONE BELOW COMPLETED ON SQLite
-------

### **VITALS 24 HR**
- This summarizes the first 24 hrs of ICU vitals for ea. patient from the 'vitalPeriodic' table

Notes:
- first 24 hours (0–1440 mins) after ICU admission
- ea. patient will have one summarized row
- mean/min/max summaries, not cleaned yet!!!

### **LABS 24 HR**
- first 24 hrs of lab tests for ea. patient from the 'lab' table

Notes:
- again only first 24 hours (0–1440 mins) after ICU admission
- normalizes common labname into its simpler names
- keeps only clinically relevant tests (glucose, Na, K, creatinine, etc.)
- ea. patient will have 1 summarized row

----------


## **Checking coverage so far**

**how many unique icu stays**

SELECT COUNT(DISTINCT patientunitstayid) AS n_patient_rows FROM patient; **-- 2520**


**how many stays have 24h vitals / labs**

SELECT COUNT(DISTINCT patientunitstayid) AS n_vitals FROM vital_24h_features; **-- ~2370**
SELECT COUNT(DISTINCT patientunitstayid) AS n_labs   FROM lab_24h_features; **-- ~2283**


**overlap between vitals and labs -- 2207**

SELECT COUNT(*) AS n_overlap
FROM (
  SELECT v.patientunitstayid
  FROM vital_24h_features v
  INNER JOIN lab_24h_features l USING (patientunitstayid)
);


**how many have APACHE patient results (mortality targets) -- 1883**

SELECT COUNT(DISTINCT patientunitstayid) AS n_apache_results FROM apachepatientresult;


**neither has labs and vitals (can drop later if needed) --74**

SELECT COUNT(*) AS n_neither
FROM patient p
LEFT JOIN vital_24h_features v USING (patientunitstayid)
LEFT JOIN lab_24h_features   l USING (patientunitstayid)
WHERE v.patientunitstayid IS NULL AND l.patientunitstayid IS NULL;

------------

## **Creating SUBSETS:** 

### **Patient Subset 'patient_subset'**
role in pipeline:
- base/dimension table for every icu stay (one row per stay)
- contains demographics and admission/discharge

why a subset: this keeps only cols used by readmission/bias/mortality merges

### **APACHE Patient Result 'apachepatientresult_subset'**
role in pipeline:
- provides predicted vs actual mortality (both icu & hospital)
- also has severity scores and provider context

why subset: can reduce to cols used for labels and fairness comparisons later

### **APACHE Patient Var 'apachepredvar_subset'**
role in pipeline:
- admission context, chronic conditions & readmission
- hospital environment (region, teachtype, managementsystem)

why subset: keep factors used in readmission and bias models

### **AdmissionDx 'admissiondx_subset'**
role in pipeline:
- high-level reason for icu admission

why subset: tryingg to keep a concise, interpretable entry for labeling 

### **NOTES AFTER CREATING SUBSETS!!**

#### **apachepatientresult (3676 rows → 2520 patients)**
- elCU stores one row per APACHE calculation per stay
- if the site ran the score multiple times (e.g., diff  versions, updates, or readmissions for same patienthealthsystemstayid)
- 2520 total icu stays in patient BUT 3676 APACHE rows → about 1.4 rows per stay, which is still normal
- KEEP IN MIND THOUGH only 1838 stays have more than one row = not all, but some repeated APACHE entries
  - **the safest way (for now i think) is to keep one APACHE row per icu stay for modeling, since we're focusing on one record per patientunitstayid**


   
#### **admissiondx (7036 rows → 2520 patients)**
- SOOO this table has one row per diagnosis term per ICU stay
  - **so a single patientunitstayid might have several entries, e.g.:sepsis, pneumonia, respiratory failure. not a duplicate, it's multi diagnosis coding**
  - what we can dooo since we only need a simple context label (for e.g. "primary diagnosis"):
    - keep the earliest or first-listed diagnosis per stay for now (making new table called 'admissiondx_one')
    - we can still go back to the full table in the db in case we want to analyze multi-diagnosis frequency or cooccurrence etc

-------
## **Creating More Tables...**

### **'apachepatientresult_one'**
da goal:
- keep one APACHE record per ICU stay (patientunitstayid) from apachepatientresult
- some stays have more than one record, will choose the most 'complete'

logic:
- rank rows within each patientunitstayid group
- rank higher if predicted fields are filled in
- IF still tied, take the LATEST apacheversion and highest id
- keep ONLY the topranked row (rn = 1)

### **'dmissiondx_one'**
goal:
- keep one admission diagnosis per ICU stay from admissiondx
- keep stay can have several diagnoses BUT we just need 1 main label

logic:     
- rank rows within ea patientunitstayid
- order by earliest admitdxenteredoffset (the time diagnosis was entered)
- if tie, keep lowest admissiondxid
- AGAIN KEEPING THE FIRST RECORD (rn = 1)

--------
## **Finally the Base Tables MERGED!!**

### **Mortality Merged Base Table**

**Key sources supporting these features:**

- Research from PMC9222812 and Nature (2020) found that basic body signs—like heart rate, blood pressure, oxygen levels, and breathing rate can be strong warning signals for risk of dying soon in the icu
- Then in PMC10796141 and Frontiers in Cardiovascular Medicine (2025), show that certain blood tests like kidney function (creatinine and BUN), and levels of sodium, potassium, and glucose are very important for predicting who is most likely to survive


**Columns Inlcuded:**

- patient demmographics:
    - age, gender, ethnicity, unittype

- sverirty/scoring: core APACHE metrics 
    - acutephysiologyscore, apachescore, apacheversion

- predicted outcomes: estimated mortality probabilities (from APACHE)
    - predictedicumortality, predictedhospitalmortality
 
- acutal outcomes: true outcomes used to assess bias and accuracy
    - actualicumortality, actualhospitalmortality
 
- early vital signs (first 24hr): key hemodynamic and respiratory measures, any abnormal HR, RR, and MAP are linked to mortality
    - hr_mean, rr_mean, sao2_mean, temp_mean, map_mean
 
- ealy labs (first 24hr): renal/metabolic stress and electrolyte imbalance, frequently associated w/ poor prognosis
    - glucose_mean, creatinine_mean, bun_mean, sodium_mean, potassium_mean

### **Readmission Base Merged Table**
**Key sources supporting these features:**

- Nature (2020) and PMC9222812 found that if your heart rate, breathing rate, or oxygen levels change a lot, you’re more likely to be sent back to the hospital after leaving the ICU
- In Frontiers in Cardiovascular Medicine (2025) says that high levels of BUN and creatinine (which show if your kidneys are under stress) mean you’re more likely to need to come back to the hospital after the ICU
- Multiple studies point out that people with health problems like diabetes, liver disease (cirrhosis), or weak immune systems are more likely to be readmitted after the ICU


**Columns Include:**
  
- demographics & icu type:
    - age, gender, ethnicity, unittype


- discharge context: poor discharge outcomes can correlate w/ higher readmission rates
    - hospitaldischargestatus, hospitaldischargelocation

  
- readmission setup : 
    - readmit, managementsystem, region

  
- underlying conditions: chronic illness burden can increasereadmission risk
    - diabetes, cirrhosis, aids, immunosuppression, metastaticcancer, lymphoma

  
- clinal context: looks at admission type or active treatment status
    - admitdiagnosis, electivesurgery, activetx

  
- vitals 24hr:
    - hr_mean, rr_mean, sao2_mean

  
- labs 24hr:
    - glucose_mean, creatinine_mean, bun_mean

### **Bias Merged Based Table**

**Key sources supporting these features:**
- PMC10796141 and PMC9222812 found that it’s important to be fair and consider people’s backgrounds like their age, gender, and race (demographics) when using AI or computer scores to judge patient risks
- Frontiers in Cardiovascular Medicine (2025) found that these clinical risk scores don’t always work equally well for everyone; for example, they may be less accurate depending on someone’s age, sex, or race in the ICU


**Columns Included:**
- demographics:
     - age, gender, ethnicity
 
- hospital context:
    - region, managementsystem

- provider context: allows bias evaluation by care specialty
    - physicianspeciality

- predicted vs actual outcomes:
    - predictedhospitalmortality, actualhospitalmortality, predictedicumortality, actualicumortality

### **ALL NOW SAVED INTO CSV**

--------